In [ ]:
import os
import subprocess
import sys
import glob

# NLGCL Kaggle Runner v7
# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Patch Code for Numpy 2.0 Compatibility
# RecBole uses deprecated aliases (np.float, np.int, etc.) removed in Numpy 2.0.
# We patch the source code specifically to support Numpy 2.x.
print("Patching RecBole for Numpy 2.x compatibility...")
files = glob.glob('recbole/**/*.py', recursive=True) + glob.glob('recbole_gnn/**/*.py', recursive=True)
replacements = {
    'np.float': 'float',
    'np.int': 'int',
    'np.object': 'object',
    'np.bool': 'bool',
    'np.str': 'str',
    'np.long': 'int'
}

patched_count = 0
for file_path in files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        new_content = content
        for old, new in replacements.items():
            # Basic textual replacement
            new_content = new_content.replace(old, new)
            
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            patched_count += 1
    except Exception as e:
        print(f"Skipping {file_path}: {e}")

print(f"Successfully patched {patched_count} files.")

# 3. Setup Dependencies
print("Configuring environment...")

# FIX: Force install Numpy 2.x and Pandas 2.2.x
# The error "Expected 96 from C header, got 88" implies some library in the Kaggle environment
# was compiled against Numpy 2.0, but we currently have Numpy 1.x active.
# We MUST upgrade to Numpy 2.0 to resolve this ABI mismatch.
print("Force upgrading to Numpy 2.0+ and Pandas 2.2+...")
!pip install "numpy>=2.0.0" "pandas>=2.2.2" --force-reinstall --upgrade

# Install PyTorch
print("Installing PyTorch 2.4.0...")
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Installing PyG wheels...")
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt>=0.2.7
torch_geometric
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing other requirements...")
!pip install -r requirements.txt

# Safe to import now
import torch
import pandas as pd
import numpy as np
print(f"Verified versions: Numpy={np.__version__}, Pandas={pd.__version__}, PyTorch={torch.__version__}")

# 4. Dataset Preprocessing
inter_path = 'dataset/QB-video/QB-video.inter'
csv_path = 'dataset/QB-video.csv'
source_path = None

if os.path.exists(csv_path):
    source_path = csv_path
elif os.path.exists(inter_path):
    source_path = inter_path

if source_path:
    print(f"Processing dataset from {source_path}...")
    df = pd.read_csv(source_path)
    
    if ':' not in str(df.columns[0]):
        print("Detected raw CSV header. Converting to RecBole format (user_id:token, item_id:token)...")
        if 'user_id' in df.columns and 'item_id' in df.columns:
            df = df[['user_id', 'item_id']]
            df.columns = ['user_id:token', 'item_id:token']
            os.makedirs(os.path.dirname(inter_path), exist_ok=True)
            df.to_csv(inter_path, index=False, sep=',')
            print(f"Successfully saved {inter_path}")
    else:
        print("Dataset header already contains type annotations.")
else:
    print("Warning: No dataset file found.")

# 5. Configuration Adjustment
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()
    if 'field_separator' not in content:
        new_content = content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\n\nfield_separator: ","')
        with open(config_path, 'w') as f:
            f.write(new_content)
        print("Updated properties/QB-video.yaml with field_separator.")

# 6. Run Training
!python main.py --dataset QB-video